In [1]:
import pandas as pd
import json
import random
import re

random.seed(42)

In [2]:
train1 = pd.read_csv("data/raw/train-00000-of-00002.csv")
train2 = pd.read_csv("data/raw/train-00001-of-00002.csv")
test = pd.read_csv("data/raw/test-00000-of-00001.csv")
info = pd.read_csv("data/raw/intent_info.csv")

print("train part 1:", train1.shape)
print("train part 2:", train2.shape)
print("test:", test.shape)
print("intent info:", info.shape)

train part 1: (5223, 3)
train part 2: (5222, 3)
test: (1400, 3)
intent info: (14, 3)


In [3]:
raw = pd.concat([train1, train2, test], ignore_index=True)
print("all rows:", raw.shape)
print(raw.isnull().sum())
raw.head()

all rows: (11845, 3)
intent_class    0
template        0
speaker_id      0
dtype: int64


,intent_class,template,speaker_id
0,6,Please tell me the IFSC Code,3
1,7,Credit card pin generation,11
2,3,Is my card dispatched?,5
3,11,I want to change the limit of my online transa...,11
4,8,Amount deducted from account without my knowledge,3


In [4]:
print("exact text:", raw["template"].nunique())

raw["template"] = raw["template"].str.strip()
print("after removing extra spaces:", raw["template"].nunique())

print("after ignoring capital letters:", raw["template"].str.lower().nunique())

exact text: 164
after removing extra spaces: 151
after ignoring capital letters: 150


In [5]:
raw["low"] = raw["template"].str.lower()
seed_df = raw[["template", "intent_class", "low"]].drop_duplicates("low")
seed_df = seed_df.drop(columns="low").reset_index(drop=True)
seed_df["template_id"] = seed_df.index

seed_df = seed_df.merge(info[["intent_class", "intent_name"]], on="intent_class", how="left")
print(seed_df.shape)
seed_df.head()

(150, 4)


,template,intent_class,template_id,intent_name
0,Please tell me the IFSC Code,6,0,ifsc_code
1,Credit card pin generation,7,1,generate_pin
2,Is my card dispatched?,3,2,dispatch_status
3,I want to change the limit of my online transa...,11,3,change_limit
4,Amount deducted from account without my knowledge,8,4,unauthorised_transaction


In [6]:
counts = seed_df["intent_class"].value_counts().sort_index()
print(counts)
print("templates with two different classes:", (raw.groupby("low")["intent_class"].nunique() > 1).sum())

intent_class
0     11
1     10
2     14
3     10
4     11
5     11
6      8
7     10
8      6
9     17
10    10
11    14
12    10
13     8
Name: count, dtype: int64
templates with two different classes: 0


In [7]:
seed_df.to_csv("data/raw/skit_unique_templates.csv", index=False)

intent_map = {}
for i, row in info.iterrows():
    intent_map[int(row["intent_class"])] = {"name": row["intent_name"], "description": row["description"]}

with open("intent_map.json", "w") as f:
    json.dump(intent_map, f, indent=2)

print("saved", len(seed_df), "templates and", len(intent_map), "intents")

saved 150 templates and 14 intents


In [8]:
has_placeholder = seed_df["template"].str.contains(r"\{\{|<|\(credit/debit\)", regex=True)
seed_df[has_placeholder][["template_id", "template", "intent_class"]]

,template_id,template,intent_class
3,3,I want to change the limit of my online transa...,11
14,14,Last <numeric> transaction details,2
27,27,I want to change the limit of my e-statement o...,11
56,56,I want to know about my last <numeric> transac...,2
62,62,I want to change the limit of my (credit/debit...,11
93,93,Tell me my last <numeric> transactions,2
112,112,{{Axis Bank}} branch location,0
126,126,List of transactions in <Month>,2
128,128,I want to change the limit of my POS transacti...,11
129,129,I want to change the limit of my international...,11


In [10]:
BANKS = ["Axis Bank", "HDFC Bank", "ICICI Bank", "SBI", "Kotak Bank"]
NUMBERS = ["2", "3", "5", "10", "five"]
MONTHS = ["January", "March", "June", "August", "November"]
CARD_TYPES = ["credit", "debit", "Visa credit", "Visa debit", "RuPay debit"]
CARD_NAMES = ["debit card", "credit card", "ATM card", "Visa card", "Mastercard"]
CREDIT_NAMES = ["credit card", "Visa credit card", "Mastercard credit card", "HDFC credit card", "SBI credit card"]
LOAN_TYPES = ["home loan", "personal loan", "car loan", "education loan", "gold loan"]

CARD_ADDONS = [" ending with 4821", " ending 7305", " issued by HDFC Bank", " issued by SBI", " issued by Axis Bank"]
ADDONS = {
    0: [" near me", " in my area", " in Pune", " in Mumbai", " in Hyderabad"],
    1: CARD_ADDONS,
    2: [" of my savings account", " of my salary account", " of my current account", " from my account", " on my account"],
    3: [" that i applied for last week", " that i requested on monday", " that i ordered recently", " from HDFC Bank", " from SBI"],
    4: [" for this month", " as of today", " for last month", " till today", " right now"],
    5: CARD_ADDONS,
    6: [" of my branch", " of my home branch", " for my savings account", " for neft transfer", " for fund transfer"],
    7: CARD_ADDONS,
    8: [" yesterday", " today", " last night", " for 5000 rupees", " of 12000 rupees"],
    9: [" for my business", " for a new house", " for 5 lakh rupees", " for my family", " for my shop"],
    10: [" of my savings account", " of my salary account", " of my current account", " today", " right now"],
    11: CARD_ADDONS,
    12: [" today", " right now", " immediately", " as soon as possible", " for security reasons"],
    13: CARD_ADDONS,
}

OPENERS = ["Hello, ", "Hi, ", "Excuse me, ", "Please help me, ", "Sir, ", "Madam, ",
           "I need help, ", "Hello sir, ", "Hello madam, ", "Kindly help me, "]
SUFFIXES = [" please", " thank you", " sir", " madam", " as soon as possible", " kindly"]
URGENT_OPENERS = ["It is urgent, ", "This is an emergency, ", "Please hurry, "]
URGENT_SUFFIXES = [" immediately", " urgently", " right now"]

FILLERS = ["please", "kindly", "the", "a", "an"]
ACRONYMS = ["ifsc", "upi", "pin", "otp", "kyc", "emi"]
SWAPS = [
    (" i want to ", " i am wanting to "),
    (" i lost ", " i have lost "),
    (" i cannot ", " i am not able to "),
    (" i am unable to ", " i am not able to "),
    (" how to ", " how can i "),
    (" what is ", " tell me what is "),
    (" is lost ", " has got lost "),
    (" is missing ", " is got missing "),
    (" details of ", " details about "),
    (" is not working ", " is not working properly "),
]

extra = pd.read_csv("data/raw/extra_phrasings.csv")
print("words lists ready, extra phrasings:", len(extra))

words lists ready, extra phrasings: 112


In [11]:
def fill_placeholders(text):
    if "{{Axis Bank}}" in text:
        return [text.replace("{{Axis Bank}}", b) for b in BANKS]
    if "<numeric>" in text:
        return [text.replace("<numeric>", n) for n in NUMBERS]
    if "<Month>" in text:
        return [text.replace("<Month>", m) for m in MONTHS]
    if "(credit/debit)" in text:
        return [text.replace("(credit/debit)", c) for c in CARD_TYPES]
    return [text]


def swap_words(text, intent):
    results = []
    if intent in [1, 3, 4, 5, 7, 11, 12, 13]:
        names = CREDIT_NAMES if intent == 4 else CARD_NAMES
        for old in ["credit card", "debit card"]:
            if old in text.lower():
                for n in names:
                    results.append(re.sub(old, n, text, flags=re.I))
    if intent == 9:
        if re.search("home loan|personal loan", text, flags=re.I):
            for t in LOAN_TYPES:
                results.append(re.sub("home loan|personal loan", t, text, flags=re.I))
        else:
            for t in LOAN_TYPES:
                results.append(re.sub(r"\bloan\b", t, text, flags=re.I))
    if intent == 0 and re.search(r"\bbank\b", text, flags=re.I):
        for b in BANKS:
            results.append(re.sub(r"\bbank\b", b, text, flags=re.I))
    return results


def lower_first(text):
    first = text.split()[0]
    if first == "I" or first.isupper() or first in ["Axis", "Kotak", "Visa", "Mastercard", "RuPay"]:
        return text
    return text[0].lower() + text[1:]


def pick(candidates, n, used, tid):
    random.shuffle(candidates)
    chosen = []
    for c in candidates:
        c = " ".join(c.split())
        key = clean(c)
        if c in used or c in chosen:
            continue
        if owner.get(key, tid) != tid:
            continue
        chosen.append(c)
        if len(chosen) == n:
            break
    return chosen

In [12]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    return " ".join(text.split())


def space_acronyms(text):
    for a in ACRONYMS:
        text = re.sub(r"\b" + a + r"\b", " ".join(a), text)
    return text


def drop_fillers(text):
    words = [w for w in text.split() if w not in FILLERS]
    return " ".join(words)


def translated_style(text):
    text = " " + text + " "
    for old, new in SWAPS:
        text = text.replace(old, new)
    return text.strip()

In [13]:
def make_bucket1(template, intent, used, tid):
    bases = fill_placeholders(template)
    first = " ".join(bases[0].split())
    candidates = list(bases)
    for b in bases[:2]:
        candidates += swap_words(b, intent)
        for a in ADDONS[intent]:
            candidates.append(b.rstrip(" ?") + a)
    others = pick(candidates, 4, used | {first}, tid)
    return [first] + others


def make_bucket2(template, intent, position, used, tid):
    base = " ".join(fill_placeholders(template)[0].split()).rstrip(" ?")
    results = []
    # one hand-written spoken sentence, if this class has one left for this template
    pool = extra[extra["intent_class"] == intent]["text"].tolist()
    if position < len(pool) and pool[position] not in used and owner.get(clean(pool[position]), tid) == tid:
        results.append(pool[position])
    openers = OPENERS + (URGENT_OPENERS if intent in [8, 12, 13] else [])
    suffixes = SUFFIXES + (URGENT_SUFFIXES if intent in [8, 12, 13] else [])
    candidates = []
    for o in openers:
        candidates.append(o + lower_first(base))
        for s in suffixes:
            if o in URGENT_OPENERS and s in URGENT_SUFFIXES:
                continue
            candidates.append(o + lower_first(base) + s)
    for s in suffixes:
        candidates.append(base + s)
    candidates = [c for c in candidates if c.lower().count("please") + c.lower().count("kindly") < 2 and c.lower().count("sir") + c.lower().count("madam") < 2]
    results += pick(candidates, 5 - len(results), used | set(results), tid)
    return results


def make_bucket3(sources, used, tid):
    noisy = []
    plain = []
    for s in sources:
        c = clean(s)
        for v in [space_acronyms(c), drop_fillers(c), translated_style(c),
                  drop_fillers(translated_style(c)), space_acronyms(drop_fillers(c))]:
            if v != c:
                noisy.append(v)
        plain.append(c)
    chosen = pick(noisy, 5, used, tid)
    if len(chosen) < 5:
        chosen += pick(plain, 5 - len(chosen), used | set(chosen), tid)
    return chosen

In [14]:
random.seed(42)
rows = []
used = set()
position_in_class = {}

# each cleaned text belongs to one template only, so train and test can never share a sentence
owner = {}
for i, t in seed_df.iterrows():
    owner[clean(fill_placeholders(t["template"])[0])] = t["template_id"]

for i, t in seed_df.iterrows():
    intent = int(t["intent_class"])
    position = position_in_class.get(intent, 0)
    position_in_class[intent] = position + 1

    tid = t["template_id"]
    b1 = make_bucket1(t["template"], intent, used, tid)
    used.update(b1)
    owner.update({clean(x): tid for x in b1})
    b2 = make_bucket2(t["template"], intent, position, used, tid)
    used.update(b2)
    owner.update({clean(x): tid for x in b2})
    b3 = make_bucket3(b1 + b2, used, tid)
    used.update(b3)
    owner.update({clean(x): tid for x in b3})

    for bucket, texts in [(1, b1), (2, b2), (3, b3)]:
        for text in texts:
            rows.append({"template_id": t["template_id"], "text": text,
                         "intent_class": intent, "intent_name": t["intent_name"], "bucket": bucket})

df = pd.DataFrame(rows)
df.insert(0, "row_id", range(len(df)))
print(df.shape)
print(df.groupby("template_id").size().value_counts())

(2250, 6)
15    150
Name: count, dtype: int64


In [15]:
print("rows:", len(df))
print("templates:", df["template_id"].nunique())
print("classes:", df["intent_class"].nunique())
print("empty text:", (df["text"].str.strip() == "").sum())
print("duplicate text rows:", df["text"].duplicated().sum())
print("rows per template:", df.groupby("template_id").size().unique())
print("unique texts after cleaning:", df["text"].apply(clean).nunique())

rows: 2250
templates: 150
classes: 14
empty text: 0
duplicate text rows: 0
rows per template: [15]
unique texts after cleaning: 2111


In [17]:
print("rows:", len(df))
print("templates:", df["template_id"].nunique())
print("classes:", df["intent_class"].nunique())
print("empty text:", (df["text"].str.strip() == "").sum())
print("duplicate text rows:", df["text"].duplicated().sum())
print("rows per template:", df.groupby("template_id").size().unique())
print("unique texts after cleaning:", df["text"].apply(clean).nunique())

rows: 2250
templates: 150
classes: 14
empty text: 0
duplicate text rows: 0
rows per template: [15]
unique texts after cleaning: 2111


In [18]:
for b in [1, 2, 3]:
    print("--- bucket", b, "---")
    print(df[df["bucket"] == b].sample(8, random_state=1)[["text", "intent_name"]].to_string(index=False))

--- bucket 1 ---
                                                                     text              intent_name
                                              Outstanding balance amount?      outstanding_balance
                     I want to know about personal loan for 5 lakh rupees               loan_query
                                   I have a uery regarding education loan               loan_query
                       I want to check my outstanding balance as of today      outstanding_balance
                                     When will you deliver my credit card          dispatch_status
I want to change the limit of my auto-debit on my credit card ending 7305             change_limit
               Fraud transaction happened from my account of 12000 rupees unauthorised_transaction
                                         How to generate pin for ATM card             generate_pin
--- bucket 2 ---
                                                                           

In [19]:
KEYWORDS = {
    0: ["branch", "address", "location", "located", "bank"],
    1: ["activate", "start", "turn on", "active"],
    2: ["transaction", "statement", "payments", "history"],
    3: ["dispatch", "deliver", "reach", "sent", "courier", "arrive", "where is my"],
    4: ["outstanding", "pending", "due", "owe", "unpaid", "bill"],
    5: ["working", "unable", "cannot", "problem", "issue", "declined", "failing", "failed", "not accepted", "trouble", "not able", "swiping"],
    6: ["ifsc", "i f s c"],
    7: ["pin", "p i n"],
    8: ["fraud", "unauthorised", "unknown", "without", "wrong", "not mine", "never made", "did not", "hacked", "complain"],
    9: ["loan"],
    10: ["balance"],
    11: ["limit"],
    12: ["block", "stop", "disable", "freeze", "deactivate"],
    13: ["lost", "missing", "misplaced", "find", "trace", "gone", "left", "wallet", "not with me"],
}

def has_keyword(text, intent):
    text = text.lower()
    for k in KEYWORDS[intent]:
        if k in text:
            return True
    return False

df["keyword_ok"] = [has_keyword(t, c) for t, c in zip(df["text"], df["intent_class"])]
print("rows with a class keyword:", df["keyword_ok"].sum(), "of", len(df))
df[~df["keyword_ok"]][["text", "intent_name"]]

rows with a class keyword: 2244 of 2250


,text,intent_name
110,how much do i need to pay on my credit card,outstanding_balance
140,how much money is there in my account,balance_enquiry
545,tell me about the money i spent last week,past_transactions
550,tell me about money i spent last week,past_transactions
860,i want to see what was debited from my account...,past_transactions
980,my debit card does not work online,card_issue


In [20]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df["text"], df["intent_class"], groups=df["template_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]
print("train rows:", len(train_df), "test rows:", len(test_df))
print("train templates:", train_df["template_id"].nunique(), "test templates:", test_df["template_id"].nunique())
print("templates in both:", len(set(train_df["template_id"]) & set(test_df["template_id"])))

train rows: 1800 test rows: 450
train templates: 120 test templates: 30
templates in both: 0


In [21]:
check = pd.DataFrame({
    "train_templates": train_df.groupby("intent_class")["template_id"].nunique(),
    "test_templates": test_df.groupby("intent_class")["template_id"].nunique(),
    "test_rows": test_df.groupby("intent_class").size(),
})
check["intent_name"] = [intent_map[c]["name"] for c in check.index]
print("classes missing from test:", 14 - test_df["intent_class"].nunique())
check

classes missing from test: 0


,train_templates,test_templates,test_rows,intent_name
intent_class,,,,
0,9,2,30,branch_address
1,8,2,30,activate_card
2,11,3,45,past_transactions
3,8,2,30,dispatch_status
4,9,2,30,outstanding_balance
5,9,2,30,card_issue
6,6,2,30,ifsc_code
7,8,2,30,generate_pin
8,5,1,15,unauthorised_transaction


In [22]:
split_table = pd.DataFrame({"template_id": seed_df["template_id"]})
split_table["split"] = "train"
split_table.loc[split_table["template_id"].isin(test_df["template_id"]), "split"] = "test"
print(split_table["split"].value_counts())

split
train    120
test      30
Name: count, dtype: int64


In [23]:
df[["row_id", "template_id", "text", "intent_class", "intent_name"]].to_csv("dataset.csv", index=False)
df.to_csv("data/raw/dataset_with_bucket.csv", index=False)
split_table.to_csv("data/raw/split_template_ids.csv", index=False)

check_df = pd.read_csv("dataset.csv")
print(check_df.shape)
print(list(check_df.columns))
check_df.head()

(2250, 5)
['row_id', 'template_id', 'text', 'intent_class', 'intent_name']


,row_id,template_id,text,intent_class,intent_name
0,0,0,Please tell me the IFSC Code,6,ifsc_code
1,1,0,Please tell me the IFSC Code for my savings ac...,6,ifsc_code
2,2,0,Please tell me the IFSC Code of my branch,6,ifsc_code
3,3,0,Please tell me the IFSC Code of my home branch,6,ifsc_code
4,4,0,Please tell me the IFSC Code for neft transfer,6,ifsc_code
